# Sentiment Analysis In-Depth
This notebook provides a deep dive into sentiment analysis using two popular libraries: TextBlob and NLTK (VADER). We will compare their performance on a comprehensive synthetic dataset of movie reviews, clean the text, and visualize the results.

# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q textblob Faker
    print('Packages installed.')

    # Download NLTK data
    import nltk
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    print('NLTK data downloaded.')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
import re
import random
from faker import Faker
from wordcloud import WordCloud

# Ensure VADER lexicon is downloaded
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)

sns.set_style('whitegrid')

## 1. Load and Prepare Data
We use `Faker` and a template system to generate a large dataset of synthetic movie reviews. This allows us to test performance on a larger scale than manual examples.

In [ ]:
fake = Faker()

def generate_reviews(n=500):
    reviews = []
    
    positive_templates = [
        "I absolutely loved this {noun}! The {aspect} was {pos_adj}.",
        "What a {pos_adj} {noun}. I would recommend it to everyone.",
        "The {aspect} was simply {pos_adj}.",
        "A masterpiece of {noun}. {pos_adj} {aspect}.",
        "I enjoyed every minute. The {aspect} was {pos_adj}."
    ]
    
    negative_templates = [
        "I hated this {noun}. The {aspect} was {neg_adj}.",
        "What a waste of time. The {noun} was {neg_adj}.",
        "Terrible {aspect}. I will never watch this again.",
        "The {aspect} was {neg_adj} and boring.",
        "Avoid this {noun} at all costs."
    ]
    
    nouns = ['movie', 'film', 'story', 'cinema']
    aspects = ['acting', 'plot', 'soundtrack', 'direction', 'screenplay']
    pos_adjs = ['superb', 'amazing', 'fantastic', 'brilliant', 'excellent', 'stunning']
    neg_adjs = ['terrible', 'awful', 'horrible', 'bad', 'disappointing', 'wooden']
    
    for _ in range(n):
        if random.random() > 0.5:
            template = random.choice(positive_templates)
            noun = random.choice(nouns)
            aspect = random.choice(aspects)
            adj = random.choice(pos_adjs)
        else:
            template = random.choice(negative_templates)
            noun = random.choice(nouns)
            aspect = random.choice(aspects)
            adj = random.choice(neg_adjs)
            
        review = template.format(noun=noun, aspect=aspect, pos_adj=adj, neg_adj=adj)
        # Occasionally add random text to make it harder
        if random.random() < 0.3:
             review += " " + fake.sentence()
             
        reviews.append(review)
        
    return reviews

reviews = generate_reviews(500)
df = pd.DataFrame(reviews, columns=['text'])
print(f"Generated {len(df)} reviews.")
print(df.head())

## 2. Text Preprocessing
Before analysis, it's good practice to clean the text. We will lowercase everything and remove special characters.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation/numbers
    return text

df['clean_text'] = df['text'].apply(clean_text)
print(df[['text', 'clean_text']].head())

## 3. TextBlob Analysis
TextBlob calculates polarity (-1 to 1) and subjectivity (0 to 1).

In [ ]:
def get_textblob_sentiment(text):
    blob = TextBlob(text)
    return blob.sentiment.polarity, blob.sentiment.subjectivity

df[['tb_polarity', 'tb_subjectivity']] = df['clean_text'].apply(
    lambda x: pd.Series(get_textblob_sentiment(x))
)
print(df[['text', 'tb_polarity', 'tb_subjectivity']].head())

## 4. VADER Analysis
VADER is rule-based and specifically tuned for social media sentiment.

In [ ]:
sia = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    # Note: VADER works best on raw text (with punctuation/caps)
    return sia.polarity_scores(text)['compound']

df['vader_compound'] = df['text'].apply(get_vader_sentiment)
print(df[['text', 'tb_polarity', 'vader_compound']].head())

## 5. Comparative Visualization
Let's compare the two methods. Do they agree?

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='tb_polarity', y='vader_compound', s=50, alpha=0.6)
plt.title('TextBlob Polarity vs VADER Compound Score')
plt.xlabel('TextBlob Polarity')
plt.ylabel('VADER Compound Score')
plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.show()

## 6. Word Clouds
Visualizing the most frequent words in positive vs negative reviews (based on VADER).

In [ ]:
positive_text = ' '.join(df[df['vader_compound'] > 0.05]['clean_text'])
negative_text = ' '.join(df[df['vader_compound'] < -0.05]['clean_text'])

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

if positive_text:
    wc_pos = WordCloud(background_color='white', colormap='Greens').generate(positive_text)
    axes[0].imshow(wc_pos, interpolation='bilinear')
    axes[0].set_title('Positive Words')
    axes[0].axis('off')

if negative_text:
    wc_neg = WordCloud(background_color='white', colormap='Reds').generate(negative_text)
    axes[1].imshow(wc_neg, interpolation='bilinear')
    axes[1].set_title('Negative Words')
    axes[1].axis('off')

plt.show()

## Conclusion
By comparing TextBlob and VADER, researchers can choose the tool that best fits their specific data type. VADER is generally more robust for short, informal text.